# Herhalingslabo Classificatie

Hoe weet je wanneer je welk algoritme best gebruikt ? Gebruik volgende handleiding:

**KNearestNeighbours**
- Geschikt voor classificatie en regressie
- Voor kleine/middelgrote datasets
- heel simpel, kan snel worden getrained
- werkt niet goed voor datasets met heel veel features

**Support Vector Machines**
- Vooral een standaard voor binaire classificatie, of wanneer je een duidelijke marge wilt op classificatie (=het hoofddoel van het algoritme)
- Voor kleine/middelgrote datasets
- Is kostelijker om te trainen dan KNN, maar classificeert dan wel sneller
- Kernel keuze en hyperparameter tuning zijn essentieel

**Decision Trees**
- Wanneer je een 'white box', 'explainable' model wilt
- kan zowel numerieke als categorische features makkelijk aan
- kan heel goed om met missende datapunten
- het risico op overfitting is groot. Bij twijfel, gebruik een random forest

**Random forest**
- Zelfde voordelen als decision tree, minder risico op overfitting
- Geschikt voor middelgrote/grote datasets
- Geschikt voor data met (heel) veel features
- Hebben uiteraard meer trainingstijd nodig dan een enkele decision tree

**Adaboost**
- Als data niet goed is gebalanceerd (niet evenveel items van elke klasse)
- Vermijd een onderfit. Dit model is beter dan 1 enkele onderliggende decision tree

**XGBoost**
- XGBoost is bijna *altijd* een goede keuze
- Middelgrote / grote datasets
- Datasets met (heel) veel features
- De implementatie van XGBoost staat, ondanks dat het een boosting algoritme is, toch deels parallellisatie toe.

In de praktijk zijn er vaak meerdere goede keuzes. Het is belangrijk te weten waarom een bepaald model wel/niet een goede keuze zou zijn, en vervolgens de meest veelbelovende modellen te proberen en te kijken wat het beste werkt.

### Wat is een kleine / middelgrote / grote dataset ?

Uiteraard is dat een beetje afhankelijk van hoelang je bereid bent te wachten. In deze cursus gebruiken we de gangbare regel:
- Klein: <10k
- Middelgroot: 10k - 100k
- Groot: >100k

Dit is ongetwijfeld over een paar jaar volledig voorbijgestreefd. Besef dat je in real life dit altijd moet afwegen tegenover de beschikbare compute power, op een Raspberry Pi 3 zal een middelgrote dataset al een stevige trainingstijd hebben, terwijl dat kinderspel is op een cloud compute server.

## Oefeningen

Behandel de volgende 3 datasets - je kan ze vnnden in de /data folder. Lees in, bekijk, en beoordeel welke algoritmes het meeste zin hebben. Pas de ML algoritmes toe en rapporteer de uiteindelijke score van je model.
- Diabetes - in de 'outcome' kolom wordt aangegeven of mensen diabetes hebben
- Customer Churn: in commerciële setting wordt vaak gesproken van 'churn', oftewel het aangeven in een soort diagram waar klanten in een process afhaken. Deze dataset bevat een 'churn' kolom die dat codeert.
- Creditcard bevat data die reeds door een ander algoritme is gegaan, waardoor de kolommen geen herkenbare naam meer hebben. De laatste kolom wijst op fraude (1) of een valide transactie (0).

### Diabetes

In [22]:
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.neighbors import KNeighborsClassifier
from sklearn.metrics import accuracy_score
from sklearn.metrics import confusion_matrix
diabetes = pd.read_csv("diabetes.csv")
# diabetes.describe() om te zien hoe groot de dataset is en hoeveel features het heeft
# Het gaat hier over maar 700+ counts dus dit telt als een kleine dataset.
# daarom koos ik voor een KNN
X,y = diabetes.drop(columns=["Outcome"]) , diabetes["Outcome"]
X_train, X_test, y_train, y_test = train_test_split(X,y,test_size=0.2,random_state=42)
scaler = StandardScaler().fit(X_train)
# ik gebruik een scaler omdat knn gevoelig is op de schaal van de features
X_train_scaled = scaler.transform(X_train)
X_test_scaled = scaler.transform(X_test)
# Ik testte verschillende waarden voor k=50. De hoogste accuracy op mijn testset was 76,62% bij k=50. 
# De resultaten tonen dat een hoger aantal buren in deze test niet automatisch tot een hogere accuracy leidt.
for k in [1, 3, 5, 7, 9, 11, 15, 25, 50, 60]:
    knn = KNeighborsClassifier(n_neighbors=k)
    knn.fit(X_train_scaled, y_train)
    y_pred = knn.predict(X_test_scaled)
    print(f"k={k}: {accuracy_score(y_test, y_pred):.2%}")

k=1: 62.34%
k=3: 70.78%
k=5: 69.48%
k=7: 68.18%
k=9: 68.18%
k=11: 70.78%
k=15: 72.08%
k=25: 74.03%
k=50: 76.62%
k=60: 75.97%


In [24]:
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.svm import SVC
from sklearn.metrics import accuracy_score
from sklearn.metrics import confusion_matrix
import matplotlib.pyplot as plt
import seaborn as sns

diabetes = pd.read_csv("diabetes.csv")
X, y = diabetes.drop(columns=["Outcome"]) , diabetes["Outcome"]

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

scaler = StandardScaler()
scaler.fit(X_train)

X_train_scaled = scaler.transform(X_train)
X_test_scaled = scaler.transform(X_test)


diabetes_svm = SVC(random_state=42)
diabetes_svm.fit(X_train_scaled, y_train)

y_pred_diabetes = diabetes_svm.predict(X_test_scaled)
accuracy_diabetes = accuracy_score(y_test, y_pred_diabetes)
print(f"Accuracy = {accuracy_diabetes*100} %")


Accuracy = 73.37662337662337 %


In [35]:
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.svm import SVC
from sklearn.metrics import accuracy_score
from sklearn.metrics import confusion_matrix
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.preprocessing import LabelEncoder
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline

churn  = pd.read_csv("Telco-Customer-Churn.csv")
# print(churn.describe()) ongeveer 7000 counts dus dit is een 
churn["TotalCharges"] = pd.to_numeric(
    churn["TotalCharges"], errors="coerce"
)

X = churn.drop(columns=["Churn", "customerID"])
y = LabelEncoder().fit_transform(churn["Churn"])

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

numeric_features = X.select_dtypes(include=["number"]).columns.tolist()
categorical_features = X.select_dtypes(include=["object"]).columns.tolist()
preprocessing = ColumnTransformer([
    ("num", Pipeline([
        ("imputer", SimpleImputer(strategy="median")),
        ("scaler", StandardScaler())
    ]), numeric_features),

    ("cat", Pipeline([
        ("imputer", SimpleImputer(strategy="most_frequent")),
        ("encoder", OneHotEncoder(handle_unknown="ignore"))
    ]), categorical_features)
])

churn_svm = Pipeline([
    ("preprocessing", preprocessing),
    ("svm", SVC())
])

churn_svm.fit(X_train, y_train)

y_pred_churn = churn_svm.predict(X_test)

accuracy_churn = accuracy_score(y_test, y_pred_churn)
print(f"Accuracy = {accuracy_churn * 100:.2f} %")

Accuracy = 80.98 %


In [ ]:
#Dit is een Random Forest
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, confusion_matrix

credit = pd.read_csv("creditcard.csv")

credit.describe()
#sinds dit een dataset is van bijna 300.000 telt dit als groot dataset met een gemiddeld aantal features, daarom ga ik het eerst proberen met
#een random forest.

X,y = credit.drop(columns=["Class"]), credit["Class"]
#ik splits de data op in X en y. X is alle data behalve Class, en y is enkel Class waarme we dan kunnen testen.
#X mag niet het resultaat al hebben daarom doen we het eruit
# alle waardes zijn precies ingevuld en rond dezelfde schaal dus ik ga het niet scalen

X_train, X_test, y_train, y_test = train_test_split(X,y,test_size=0.2,random_state=42)
#ik splits hier de data op in de standaard 80/20 en gebruikt random_state 42
random = RandomForestClassifier(n_estimators=20,random_state=42)
#ik gebruik de standaard 100 n_estimators en opnieuw random_state 42
random.fit(X_train,y_train)
#hier fit ik het model

y_pred_random = random.predict(X_test)
#hier test ik de accuracy
accuracy_random = accuracy_score(y_test,y_pred_random)
#en hier vergelijk ik de accuracy met mijn model en de echte waarde
print(accuracy_random)
#omdat ik een heel hoge accuracy kreeg ga ik ook de precisie tonen en een confusion matrix tonen om te controleren
precision_random = precision_score(y_test, y_pred_random)

confusion_random = confusion_matrix(y_test, y_pred_random)
print(f"Precision: {precision_random:.2f}")
print("Confusion Matrix:")
print(confusion_random)

0.9995259997893332
Precision: 0.97
Confusion Matrix:
[[56862     2]
 [   25    73]]


In [ ]:
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, precision_score, confusion_matrix
from xgboost import XGBClassifier


credit = pd.read_csv("creditcard.csv")

credit.describe()
#Deze keer ga ik XGBoost proberen.

X,y = credit.drop(columns=["Class"]), credit["Class"]
#ik splits de data op in X en y. X is alle data behalve Class, en y is enkel Class waarme we dan kunnen testen.
#X mag niet het resultaat al hebben daarom doen we het eruit
# alle waardes zijn precies ingevuld en rond dezelfde schaal dus ik ga het niet scalen

X_train, X_test, y_train, y_test = train_test_split(X,y,test_size=0.2,random_state=42)
#ik splits hier de data op in de standaard 80/20 en gebruikt random_state 42
xgb = XGBClassifier(
        n_estimators=100,
        learning_rate = 0.1,
        random_state = 42
)
#Net zoals in de les gebruik ik 0.1 learning rate maar ik doe geen max_depth voor nu
xgb.fit(X_train,y_train)
#hier fit ik het model

y_pred_xgb = xgb.predict(X_test)
#hier test ik de accuracy
accuracy_xgb = accuracy_score(y_test,y_pred_xgb)
#en hier vergelijk ik de accuracy met mijn model en de echte waarde
print(accuracy_xgb)
#Om het goed te kunnen vergelijken met de Random Forest gebruik ik hier ook een precision en toon ik een confusion matrix
#zo is het gemakkelijker om te zien welke beter is
precision_xgb = precision_score(y_test, y_pred_xgb)

confusion_xgb = confusion_matrix(y_test, y_pred_xgb)
print(f"Precision: {precision_xgb:.2f}")
print("Confusion Matrix:")
print(confusion_xgb)



0.9995786664794073
Precision: 0.97
Confusion Matrix:
[[56862     2]
 [   22    76]]


Time      0
V1        0
V2        0
V3        0
V4        0
V5        0
V6        0
V7        0
V8        0
V9        0
V10       0
V11       0
V12       0
V13       0
V14       0
V15       0
V16       0
V17       0
V18       0
V19       0
V20       0
V21       0
V22       0
V23       0
V24       0
V25       0
V26       0
V27       0
V28       0
Amount    0
Class     0
dtype: int64